<a href="https://colab.research.google.com/github/Bhanvi08/ML-Predictive-Modeling-Suite/blob/main/SUPERVISED_LEARNING_by_Bhanvi_Singh_Chauhan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Name: Bhanvi Singh Chauhan
# Imbalance Learning - SUPERVISED LEARNING

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import RandomOverSampler
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA
url = 'https://raw.githubusercontent.com/AnjulaMehto/Sampling_Assignment/main/Creditcard_data.csv'
df = pd.read_csv(url)

# 2. BALANCE THE DATASET
# 'Class' is the target (0 for normal, 1 for fraud)
X = df.drop('Class', axis=1)
y = df['Class']

# Using RandomOverSampler to make it 50/50 (Balancing the classes)
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
balanced_df = pd.concat([X_resampled, y_resampled], axis=1)

print(f"Balanced Dataset Size: {balanced_df.shape}")

# 3. CALCULATE SAMPLE SIZE (n)
# Formula: n = (Z^2 * p * (1-p)) / e^2
# For 95% confidence (Z=1.96), p=0.5, e=0.05
Z = 1.96
p = 0.5
e = 0.05
n = int(np.ceil((Z**2 * p * (1-p)) / (e**2)))
print(f"Calculated Sample Size (n) based on formula: {n}")

# 4. DEFINE 5 SAMPLING TECHNIQUES
def get_samples(data, n):
    samples = []
    # Sampling 1: Simple Random Sampling
    samples.append(data.sample(n=n, random_state=42))
    # Sampling 2: Systematic Sampling
    interval = len(data) // n
    samples.append(data.iloc[::interval][:n])
    # Sampling 3: Stratified Sampling
    samples.append(data.groupby('Class', group_keys=False).apply(lambda x: x.sample(n // 2, random_state=42)))
    # Sampling 4: Cluster Sampling
    # Creating 10 clusters
    data['Cluster'] = np.repeat(range(10), len(data)/10 + 1)[:len(data)]
    selected_clusters = [1, 5, 8] # Random clusters
    samples.append(data[data['Cluster'].isin(selected_clusters)].sample(n=n, random_state=42).drop('Cluster', axis=1))
    data.drop('Cluster', axis=1, inplace=True)
    # Sampling 5: Convenience Sampling
    samples.append(data.head(n))
    return samples

samples = get_samples(balanced_df, n)

# 5. DEFINE 5 MODELS
models = [
    LogisticRegression(max_iter=2000),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    SVC(random_state=42),
    GaussianNB()
]

model_names = ['M1 (Logistic)', 'M2 (DecisionTree)', 'M3 (RandomForest)', 'M4 (SVM)', 'M5 (NaiveBayes)']
sampling_names = ['Sampling1', 'Sampling2', 'Sampling3', 'Sampling4', 'Sampling5']

# 6. EVALUATE MODELS ON SAMPLES
results = np.zeros((5, 5))

for i, sample in enumerate(samples):
    X_s = sample.drop('Class', axis=1)
    y_s = sample['Class']
    # Split each sample into training and testing
    X_train, X_test, y_train, y_test = train_test_split(X_s, y_s, test_size=0.2, random_state=42)

    for j, model in enumerate(models):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        results[j, i] = accuracy_score(y_test, y_pred) * 100

# 7. CREATE FINAL TABLE
final_table = pd.DataFrame(results, index=model_names, columns=sampling_names)
print("\n--- Accuracy Result Table (%) ---")
print(final_table)

# Determine best
best_coord = np.unravel_index(np.argmax(results, axis=None), results.shape)
print(f"\nHighest Accuracy: {results[best_coord]:.2f}% using {model_names[best_coord[0]]} and {sampling_names[best_coord[1]]}")

Balanced Dataset Size: (1526, 31)
Calculated Sample Size (n) based on formula: 385

--- Accuracy Result Table (%) ---
                   Sampling1   Sampling2  Sampling3   Sampling4   Sampling5
M1 (Logistic)      89.610390   88.311688  90.909091   96.103896  100.000000
M2 (DecisionTree)  97.402597   97.402597  98.701299   98.701299   96.103896
M3 (RandomForest)  98.701299  100.000000  98.701299  100.000000  100.000000
M4 (SVM)           70.129870   75.324675  74.025974   84.415584  100.000000
M5 (NaiveBayes)    70.129870   83.116883  66.233766   96.103896   96.103896

Highest Accuracy: 100.00% using M1 (Logistic) and Sampling5
